# L5 · Papel de trabajo (Excel)

**Criterio (docs/13):** el Excel abre sin advertencias; los conteos de la Hoja 1 cuadran con la Hoja 2; cero llamadas al modelo al generarlo.

**Qué es real y qué es entrada de prueba.** Los textos son reales (PDF de normas de la SB y manual MOCK, leídos con PyMuPDF) y `Seccion`, `Par`, `Veredicto` son los modelos reales de `backend/models/schemas.py`. La marca de cada artículo y los scores los fija un **guion determinista** (`scripts/datos_papel.py`): son la *entrada* que L4 entregará, no una simulación de un modelo. El borrador de conclusión **sí** usa el LLM real (Groq, stage sandbox) con `backend/prompts/conclusion.md`.

In [ ]:
# --- Arranque por stage (ver 00_modelos.ipynb) ---------------------------------------------------
# Local: el stage sale de config/.env (STAGE=dev MacBook+DMR · STAGE=sandbox Raspberry+Groq+Ollama). Nada que hacer.
# Databricks (STAGE=mvp): descomenta y ejecuta ANTES de importar `backend`.
# %pip install -r ../requirements.txt
# dbutils.library.restartPython()
# import os
# os.environ["STAGE"] = "mvp"
# os.environ["FOUNDRY_AI_TOKEN"] = dbutils.secrets.get(scope="comparador", key="foundry-token")


In [1]:
import os, sys, time, json, zipfile, subprocess, warnings, socket
from collections import Counter
from pathlib import Path
os.environ.setdefault("STAGE", "sandbox")
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
import pymupdf
from backend.core.progreso import configurar_logs   # convención: log sencillo
configurar_logs()
from openpyxl import load_workbook

from backend.config import settings, falta
from backend.llm.client import ModelClient
from backend.output import annex, styles, workpaper
from backend.output.conclusion import generar_borrador, resumen_conclusion, cifras_no_entregadas
from backend.output.workpaper import generar_papel, conteos_papel, secciones_excluidas, COLUMNAS, PREFIJO_BORRADOR
from scripts.datos_papel import armar_entrada, cargar_manual, cargar_norma, documento, texto_fuente, hostilizar

from backend.config import carpeta_normas, carpeta_manuales
NORMAS = carpeta_normas()   # documentos/normas/ (no se versiona); cambia con NORMAS_DIR
if not NORMAS.exists():
    print(f"⚠ Falta la carpeta {NORMAS}: copia ahí los PDF de las normas (ver documentos/LEEME.md)")
MANUAL = carpeta_manuales() / os.environ.get("MANUAL_PRUEBA", "MOCK-DEMO-01.pdf")   # MANUAL_PRUEBA=MANUAL-ARLAFT-V14.pdf para un manual local
BLOQUES_CACHE = Path(os.environ["BLOQUES_DIR"]) if os.environ.get("BLOQUES_DIR") else None
SALIDA = RAIZ / "output"; SALIDA.mkdir(exist_ok=True)      # carpeta ignorada por git
print("stage:", os.environ["STAGE"], "| LLM:", settings.LLM_PROVIDER, settings.GROQ_LLM_MODEL,
      "| clave configurada:", not falta(settings.GROQ_API_KEY))
print("colores del papel:", settings.COLOR_PRIMARIO, settings.COLOR_FONDO)

stage: sandbox | LLM: groq openai/gpt-oss-120b | clave configurada: True
colores del papel: 1E7B4F FFFFFF


## 1 · Entrada con textos reales
Tres capítulos de la norma de la SB (L1-XVI, caps. III, IV y V) contra el manual MOCK-DEMO-01.

In [2]:
pdfs_norma = [NORMAS / f"L1-XVI-cap-{c}.pdf" for c in ("III", "IV", "V")]
normas = [(documento(p, f"N{i}", "normativa"), cargar_norma(p, f"N{i}")) for i, p in enumerate(pdfs_norma, 1)]
excluidas_papel = []
if MANUAL.name.startswith("MOCK-DEMO-"):
    secs_manual = cargar_manual(MANUAL, "M1")
else:                                  # manual local: Docling real + cascada de seccionado; solo las hojas se comparan
    from backend.core.progreso import progreso
    from backend.extraction.lector import leer_documento
    from backend.retrieval import seleccion
    from backend.retrieval.tabla import tabla_recuperacion
    with progreso(f"Docling · {MANUAL.name}", "pág") as cb:
        todas_m = leer_documento(MANUAL, "M1", "manual_control", progreso=cb, cache=BLOQUES_CACHE)
    t_man = tabla_recuperacion(todas_m, MANUAL, MANUAL.stem)
    secs_manual = seleccion.solo_incluidas([s for s in todas_m if s.es_hoja], t_man)     # sin carátula, preámbulo, índice ni anexos
    excluidas_papel = secciones_excluidas(seleccion.excluidas(t_man))                    # van al anexo (Hoja 2) con su motivo
    print(f"Secciones excluidas como no comparables: {len(excluidas_papel)} (quedan {len(secs_manual)} para comparar)")
manual = (documento(MANUAL, "M1", "manual_control"), secs_manual)
entrada = armar_entrada(normas, manual).model_copy(update={"excluidas": excluidas_papel})
print("normas:", [(d.id, d.nombre, len(s)) for d, s in normas])
print("manual:", manual[0].nombre, "-", len(manual[1]), "secciones")
print("filas:", len(entrada.filas), "| pares:", len(entrada.pares), "| controles sin base:", len(entrada.controles_sin_base))
print("marcas de entrada:", conteos_papel(entrada))
# texto literal: cada artículo y sección es subcadena del texto del PDF (espacios normalizados)
fuentes = {d.id: texto_fuente(p) for (d, _), p in zip(normas, pdfs_norma)} | {"M1": texto_fuente(MANUAL)}
todas = [a for _, s in normas for a in s] + manual[1]
fuera = [s.id for s in todas if " ".join(s.texto_literal.split()) not in fuentes[s.doc_id]]
print("secciones cuyo texto NO es subcadena del PDF:", fuera)
if MANUAL.name.startswith("MOCK-DEMO-"):
    assert not fuera
else:    # en un manual real las tablas salen como Markdown y la numeración se antepone: no coinciden letra a letra con el PDF
    print(f"(informativo) {len(fuera)} de {len(todas)} secciones no son subcadena literal del PDF por tablas y numeración")

normas: [('N1', 'L1-XVI-cap-III.pdf', 8), ('N2', 'L1-XVI-cap-IV.pdf', 15), ('N3', 'L1-XVI-cap-V.pdf', 5)]
manual: MOCK-DEMO-01.pdf - 21 secciones
filas: 28 | pares: 84 | controles sin base: 8
marcas de entrada: {'A': 11, 'L': 6, 'R': 6, 'X': 2, 'P': 3}


secciones cuyo texto NO es subcadena del PDF: []


## 2 · Borrador de conclusión: UNA llamada real a Groq
El resumen (conteos, totales, artículos R y L, controles sin base) se calcula en Python. El código verifica 120–220 palabras y que el texto no traiga cifras fuera del resumen.

In [3]:
class ContadorCliente(ModelClient):
    """El cliente real de siempre, pero cuenta cada llamada al LLM y a embeddings."""
    llamadas_llm = 0
    llamadas_emb = 0
    def _completar(self, mensajes):
        self.llamadas_llm += 1
        return super()._completar(mensajes)
    def embed_multi(self, textos, progreso=None):
        self.llamadas_emb += 1
        return super().embed_multi(textos, progreso)

cliente = ContadorCliente()
resumen = resumen_conclusion(entrada)
print(json.dumps(resumen, ensure_ascii=False, indent=1)[:1500])
t0 = time.perf_counter()
borrador = generar_borrador(entrada, cliente)
t_llm = time.perf_counter() - t0
print(f"\nllamadas al LLM: {cliente.llamadas_llm} | {len(borrador.split())} palabras | {t_llm:.1f}s")
print("cifras que no están en el resumen:", cifras_no_entregadas(borrador, resumen))
print("\n" + borrador)
assert cliente.llamadas_llm in (1, 2) and 120 <= len(borrador.split()) <= 220   # 2 si el primer borrador no validó y se reintentó

{
 "manual": "MOCK-DEMO-01.pdf",
 "normativas": [
  "L1-XVI-cap-III.pdf",
  "L1-XVI-cap-IV.pdf",
  "L1-XVI-cap-V.pdf"
 ],
 "total_articulos": 28,
 "conteos_por_marca": {
  "A": 11,
  "L": 6,
  "R": 6,
  "X": 2,
  "P": 3
 },
 "articulos_evaluables_A_L_R": 23,
 "porcentaje_por_marca_sobre_total": {
  "A": 39.3,
  "L": 21.4,
  "R": 21.4,
  "X": 7.1,
  "P": 10.7
 },
 "porcentaje_cumple_sobre_evaluables": 47.8,
 "porcentaje_parcial_sobre_evaluables": 26.1,
 "porcentaje_no_cumple_sobre_evaluables": 26.1,
 "articulos_R": [
  {
   "identificador": "Art. 3",
   "elementos_faltantes": [
    "Procedimiento equivalente a Art. 3"
   ]
  },
  {
   "identificador": "Art. 7",
   "elementos_faltantes": [
    "Procedimiento equivalente a Art. 7"
   ]
  },
  {
   "identificador": "Art. 5",
   "elementos_faltantes": [
    "Procedimiento equivalente a Art. 5"
   ]
  },
  {
   "identificador": "Art. 9",
   "elementos_faltantes": [
    "Procedimiento equivalente a Art. 9"
   ]
  },
  {
   "identificador": "A


llamadas al LLM: 1 | 152 palabras | 7.5s
cifras que no están en el resumen: []

En el marco de la auditoría del manual MOCK-DEMO-01.pdf, se evaluaron 28 artículos contenidos en las normativas L1-XVI-cap-III.pdf, L1-XVI-cap-IV.pdf y L1-XVI-cap-V.pdf. De los 23 artículos evaluables, el nivel general de cumplimiento se sitúa en 47.8% de los artículos, con cumplimiento parcial en 26.1% y no cumplimiento en 26.1%. Los incumplimientos (R) más relevantes se identifican en Art. 3, Art. 7, Art. 5, Art. 9, Art. 15 y Art. 4, donde falta el procedimiento equivalente correspondiente. Los cumplimientos parciales (L) se observan en Art. 2, Art. 6, Art. 4, Art. 8, Art. 14 y Art. 3, por ausencia de plazo o responsable especificado. Además, se detectaron ocho controles sin base normativa (III, IV, 5.4, VI, VIII, X, XI, XIII). Estos hallazgos indican áreas críticas que requieren la incorporación de los procedimientos y asignaciones faltantes, así como la alineación de los controles identificados con la 

### Validador puro y 5 corridas reales del borrador (QA-03)
`validar_borrador(texto, resumen)` es pura (se prueba sin modelo). Aquí se corre 5 veces contra Groq real y se imprime cada resultado literal; la validación NO se relaja.

In [4]:
from backend.config import cargar_prompt, rellenar_prompt
from backend.core.errors import LLMOutputError
from backend.output.conclusion import validar_borrador, BorradorConclusion
# casos puros, sin modelo
assert cifras_no_entregadas("47,8 % de 28", resumen) == [] and cifras_no_entregadas("16,7 %", resumen) == [16.7]
for texto, esperado in [("palabra " * 50, "palabras"), ("palabra " * 150 + "con 16,7 %", "cifras")]:
    try: validar_borrador(texto, resumen); print("¡aceptado!")
    except LLMOutputError as e: print("rechazo esperado:", str(e)[:70]); assert esperado in str(e)
ok_runs = 0
from tqdm import tqdm   # barras de progreso
barra = tqdm(range(1, 6), desc="Borrador de conclusión (LLM)", unit="corrida")
for n in barra:
    c = ContadorCliente()
    sistema, usuario = cargar_prompt("conclusion")
    usuario = rellenar_prompt(usuario, resumen_json=json.dumps(resumen, ensure_ascii=False, indent=1))
    texto = c.chat_json(sistema, usuario, BorradorConclusion).borrador_conclusion
    try:
        validar_borrador(texto, resumen); estado = "VÁLIDO"; ok_runs += 1
    except LLMOutputError as e:
        estado = f"RECHAZADO: {e}"
    tqdm.write(f"--- corrida {n}: llamadas={c.llamadas_llm} palabras={len(texto.split())} cifras ajenas={cifras_no_entregadas(texto, resumen)} -> {estado}")
    tqdm.write(texto)
print("\nválidas:", ok_runs, "de 5")

rechazo esperado: Borrador de 50 palabras (se piden 120-220)
rechazo esperado: El borrador trae cifras que no están en el resumen: [16.7]


--- corrida 1: llamadas=1 palabras=154 cifras ajenas=[] -> VÁLIDO
Se concluye que, de los 28 artículos revisados en el manual MOCK-DEMO-01.pdf y las tres normativas L1-XVI-cap-III.pdf, L1-XVI-cap-IV.pdf y L1-XVI-cap-V.pdf, 23 artículos fueron evaluables. El porcentaje de cumplimiento sobre los evaluables es 47.8%, el de cumplimiento parcial 26.1% y el de no cumplimiento 26.1%. Los artículos con marca R, que indican incumplimiento, son Art. 3, Art. 4, Art. 5, Art. 7, Art. 9 y Art. 15, cada uno con la ausencia del procedimiento equivalente correspondiente. Los artículos con marca L, que indican cumplimiento parcial, son Art. 2, Art. 3, Art. 4, Art. 6, Art. 8 y Art. 14, por falta de plazo o responsable definido. Además, se identificaron ocho controles sin base normativa (III, IV, 5.4, VI, VIII, X, XI, XIII). En conjunto, el nivel general de cumplimiento es moderado, con áreas críticas que requieren la incorporación de los procedimientos y la definición de responsables para alcanzar la ple

--- corrida 2: llamadas=1 palabras=151 cifras ajenas=[] -> VÁLIDO
Se concluye que, de los 28 artículos analizados en el manual MOCK-DEMO-01.pdf y las tres normativas L1-XVI-cap-III.pdf, L1-XVI-cap-IV.pdf y L1-XVI-cap-V.pdf, 23 son evaluables según los criterios de cumplimiento. El nivel general de cumplimiento se sitúa en 47.8 % sobre los artículos evaluables, con 26.1 % de cumplimiento parcial y 26.1 % de no cumplimiento. Los incumplimientos (R) se concentran en los Art. 3, Art. 7, Art. 5, Art. 9, Art. 15 y Art. 4, donde falta el procedimiento equivalente correspondiente. Los cumplimientos parciales (L) se observan en los Art. 2, Art. 6, Art. 4, Art. 8, Art. 14 y Art. 3, por ausencia de plazo o responsable definido. Además, se identificaron ocho controles sin base normativa (III, IV, 5.4, VI, VIII, X, XI, XIII). Estos hallazgos indican la necesidad de reforzar los procedimientos faltantes y de formalizar los controles no respaldados por la normativa para elevar el grado de cumplimient

--- corrida 3: llamadas=1 palabras=154 cifras ajenas=[] -> VÁLIDO
Se ha evaluado el manual MOCK-DEMO-01.pdf frente a las normativas L1-XVI-cap-III.pdf, L1-XVI-cap-IV.pdf y L1-XVI-cap-V.pdf, abarcando un total de 28 artículos, de los cuales 23 son evaluables (A, L, R). El nivel general de cumplimiento se refleja en un 47.8 % de los artículos evaluables, mientras que el 26.1 % presenta cumplimiento parcial y el 26.1 % no cumple. Los resultados de la marca A ascienden a 11 artículos (39.3 % del total), L a 6 artículos (21.4 %) y R a 6 artículos (21.4 %). Los incumplimientos críticos (R) se identifican en los Art. 3, Art. 7, Art. 5, Art. 9, Art. 15 y Art. 4, por ausencia de procedimientos equivalentes. Los cumplimientos parciales (L) se observan en los Art. 2, Art. 6, Art. 4, Art. 8, Art. 14 y Art. 3, por falta de plazo o responsable. Además, se detectaron ocho controles sin base normativa (III, IV, 5.4, VI, VIII, X, XI, XIII).


--- corrida 4: llamadas=1 palabras=155 cifras ajenas=[] -> VÁLIDO
Se concluye que, de los 28 artículos revisados del manual MOCK-DEMO-01.pdf y de las tres normativas L1-XVI-cap-III.pdf, L1-XVI-cap-IV.pdf y L1-XVI-cap-V.pdf, 23 artículos fueron evaluables (A, L o R). El nivel general de cumplimiento se sitúa en 47.8% sobre los evaluables, con 39.3% de los artículos marcados como A, 21.4% como L y 21.4% como R. Los incumplimientos (R) se observaron en los artículos 3, 7, 5, 9, 15 y 4, donde falta el procedimiento equivalente correspondiente. Los cumplimientos parciales (L) se registraron en los artículos 2, 6, 4, 8, 14 y 3, por ausencia de plazo o responsable. Además, se identificaron ocho controles sin base normativa (III, IV, 5.4, VI, VIII, X, XI, XIII). No se detectaron artículos adicionales R ni L fuera de los listados. La evidencia recopilada permite afirmar que, si bien existe una proporción significativa de cumplimiento, persisten áreas críticas que requieren la incorporación de l

--- corrida 5: llamadas=1 palabras=161 cifras ajenas=[] -> VÁLIDO
El presente trabajo de auditoría se basa en la revisión del manual MOCK-DEMO-01.pdf y de las normativas L1-XVI-cap-III.pdf, L1-XVI-cap-IV.pdf y L1-XVI-cap-V.pdf. Sobre un total de 28 artículos, 23 fueron evaluables bajo los criterios de cumplimiento (A), cumplimiento parcial (L) y no cumplimiento (R). El nivel general de cumplimiento se refleja en un 47.8 % de los artículos evaluables, mientras que el 26.1 % presenta cumplimiento parcial y el 26.1 % no cumple. Se registraron 11 artículos con cumplimiento total (A) y 6 con cumplimiento parcial (L). Los incumplimientos más relevantes (R) corresponden a los Art. 3, Art. 7, Art. 5, Art. 9, Art. 15 y Art. 4, cada uno con la ausencia del procedimiento equivalente indicado. Los cumplimientos parciales (L) más significativos se observaron en los Art. 2, Art. 6, Art. 4, Art. 8, Art. 14 y Art. 3, donde faltan plazo o responsable específico. Además, se identificaron ocho controles 

## 3 · Generar el papel y demostrar que no hay llamadas al modelo
Se cuentan las llamadas del cliente alrededor de `generar_papel` y, además, se bloquea toda conexión de red durante la generación.

In [5]:
entrada.conclusion = borrador          # sin editar: el papel añade el prefijo
llm0, emb0 = cliente.llamadas_llm, cliente.llamadas_emb
conexiones = []
conectar_real = socket.socket.connect
def vigilar(self, *a, **k):
    conexiones.append(a); raise RuntimeError("red bloqueada durante generar_papel")
socket.socket.connect = vigilar
try:
    t0 = time.perf_counter()
    conteos = generar_papel(entrada, SALIDA / "papel_l5.xlsx")
    t_gen = time.perf_counter() - t0
finally:
    socket.socket.connect = conectar_real
print(f"generar_papel: {t_gen:.2f}s | conteos: {conteos}")
print("llamadas LLM durante la generación:", cliente.llamadas_llm - llm0, "| embeddings:", cliente.llamadas_emb - emb0,
      "| intentos de conexión:", len(conexiones))
import re
for m in (annex, styles, workpaper):
    imports = re.findall(r"^\s*(?:from|import)\s+(\S+)", Path(m.__file__).read_text(), re.M)
    print(m.__name__, "| importa llm/openai:", any(i.startswith(("backend.llm", "openai")) for i in imports))
assert cliente.llamadas_llm == llm0 and cliente.llamadas_emb == emb0 and not conexiones

generar_papel: 0.25s | conteos: {'A': 11, 'L': 6, 'R': 6, 'X': 2, 'P': 3}
llamadas LLM durante la generación: 0 | embeddings: 0 | intentos de conexión: 0
backend.output.annex | importa llm/openai: False
backend.output.styles | importa llm/openai: False
backend.output.workpaper | importa llm/openai: False


## 4 · Recargar el archivo y revisar el criterio de docs/10

In [6]:
ARCHIVO = SALIDA / "papel_l5.xlsx"
with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    wb = load_workbook(ARCHIVO)                      # openpyxl: sin advertencias de formato
h1, h2 = wb["Papel de trabajo"], wb["Anexo técnico"]
print("hojas:", wb.sheetnames, "| advertencias al abrir:", len(avisos))
print("encabezado de Hoja 1:")
for r in range(1, 11):
    print(f"  {h1.cell(r,1).value:<28}|", (str(h1.cell(r,2).value) or "").replace("\n", " / ")[:95])

hojas: ['Papel de trabajo', 'Anexo técnico'] | advertencias al abrir: 0
encabezado de Hoja 1:
  Nombre de la revisión       | None
  Nombre del papel de trabajo | Evaluación de cumplimiento normativo — MOCK-DEMO-01.pdf
  Objetivo                    | Evaluar el cumplimiento de la normativa regulatoria aplicable por parte del manual de control i
  Corte o periodo             | None
  Fuente                      | L1-XVI-cap-III.pdf: Libro I › Título XVI › Capítulo III / L1-XVI-cap-IV.pdf: Libro I › Título X
  Marcas de verificación      | A = Sí cumple  ·  L = Parcialmente  ·  R = No cumple  ·  X = No aplica  ·  P = Información obte
  Conclusiones                | [Borrador generado automáticamente — validar] / En el marco de la auditoría del manual MOCK-DEM
  Elaborado por               | None
  Revisado por                | None
  Fecha de ejecución          | 07/10/2026


In [7]:
FILA_H = 12
esperadas = ["Nombre de la normativa", "Sección / Artículo", "Manual interno", "Verificación", "Comentario", "Referencias / Evidencias"]
cabecera = [h1.cell(FILA_H, c).value for c in range(1, 7)]
filas1 = [r for r in range(FILA_H + 1, h1.max_row + 1) if h1.cell(r, 4).value]
ley = h1.cell(6, 2).value
print("columnas en el orden exacto:", cabecera == esperadas)
print("leyenda:", ley)
print("filas de la matriz:", len(filas1))
print("anchos:", [h1.column_dimensions[c].width for c in "ABCDEF"], "| congelado:", h1.freeze_panes)
print("encabezado: fondo", h1.cell(FILA_H, 1).fill.start_color.rgb, "texto", h1.cell(FILA_H, 1).font.color.rgb, "negrita", h1.cell(FILA_H, 1).font.b)
print("celdas combinadas en col. 1:", sum(1 for m in h1.merged_cells.ranges if m.min_col == 1 == m.max_col and m.min_row > FILA_H))
print("color por marca:", {h1.cell(r, 4).value: h1.cell(r, 4).fill.start_color.rgb for r in reversed(filas1)})
print("wrap_text en la matriz:", all(h1.cell(r, c).alignment.wrap_text for r in filas1 for c in range(2, 7)))
print("ruta con ›:", h1.cell(filas1[0], 2).value.split("\n")[0])
print("revisión y corte en blanco:", repr(h1["B1"].value), repr(h1["B4"].value), "| elaborado/revisado:", repr(h1["B8"].value), repr(h1["B9"].value))
print("Conclusiones empieza por:", h1["B7"].value[:60])

columnas en el orden exacto: True
leyenda: A = Sí cumple  ·  L = Parcialmente  ·  R = No cumple  ·  X = No aplica  ·  P = Información obtenida de la normativa
filas de la matriz: 28
anchos: [18.0, 45.0, 45.0, 10.0, 40.0, 28.0] | congelado: A13
encabezado: fondo 001E7B4F texto 00FFFFFF negrita True
celdas combinadas en col. 1: 3
color por marca: {'A': '00C6EFCE', 'R': '00F8CBAD', 'L': '00FFEB9C', 'P': '00DDEBF7', 'X': '00E7E6E6'}
wrap_text en la matriz: True
ruta con ›: Capítulo III › Sección I › Art. 1
revisión y corte en blanco: None None | elaborado/revisado: None None
Conclusiones empieza por: [Borrador generado automáticamente — validar]
En el marco de


In [8]:
# --- texto literal: col. 2 = ruta + texto del artículo; col. 3 = ruta + cita del manual
malos, marcados = [], 0
ident_en_fila = [h1.cell(r, 2).value.split("\n", 1)[0].split(" › ")[-1] for r in filas1]
esperado = [f.articulo.identificador for f in sorted(entrada.filas, key=lambda f: (f.articulo.doc_id, f.articulo.pagina_inicio, f.articulo.id))]
print("orden = documento normativo:", ident_en_fila == esperado)
for r, f in zip(filas1, sorted(entrada.filas, key=lambda f: (f.articulo.doc_id, f.articulo.pagina_inicio, f.articulo.id))):
    texto = h1.cell(r, 2).value.split("\n", 1)[1]
    if texto != f.articulo.texto_literal and not texto.endswith("]"): malos.append(("norma", r))
    c3 = h1.cell(r, 3).value or ""
    if "\n" in c3:
        cita = c3.split("\n", 1)[1]
        if cita not in f.respaldo.texto_literal:
            if "Aviso" in (h1.cell(r, 6).value or ""): marcados += 1
            else: malos.append(("manual", r))
    if f.marca in ("R", "X", "P") and c3: malos.append(("manual no vacío en R/X/P", r))
print("textos que no son subcadena del texto fuente y sin aviso:", malos, "| con aviso:", marcados)
assert not malos

orden = documento normativo: True
textos que no son subcadena del texto fuente y sin aviso: [] | con aviso: 0


In [9]:
# --- conteos: Hoja 1 = conteos DERIVABLES de la Hoja 2.
# Chequeo solo del notebook (no es código del backend): se re-deriva la marca de cada artículo con las
# reglas de docs/09 §5 aplicadas a las columnas de la Hoja 2 (naturaleza, cobertura, confianza, elegido).
fila_via2 = next(r for r in range(1, h2.max_row + 1) if str(h2.cell(r, 1).value or "").startswith("Vía 2"))
filas2 = [r for r in range(2, fila_via2) if h2.cell(r, 1).value]
enc2 = [h2.cell(1, c).value for c in range(1, h2.max_column + 1)]
col = {n: enc2.index(n) + 1 for n in enc2}
pares_h2 = {}
for r in filas2:
    art = h2.cell(r, col["Artículo"]).value.split(" | ")[0]
    pares_h2.setdefault(art, []).append({k: h2.cell(r, col[k]).value for k in ("Naturaleza", "Cobertura", "Confianza", "Elegido para el papel")})

def marca_derivada(pares):
    validos = [p for p in pares if p["Naturaleza"]]
    n = Counter(p["Naturaleza"] for p in validos)
    nat = "obligacion" if n["obligacion"] >= max(n.values()) else n.most_common(1)[0][0]   # empate -> obligación
    if nat == "informativo": return "P"
    if nat == "no_aplica_entidad": return "X"
    if any(p["Cobertura"] == "total" for p in validos): return "A"
    if any(p["Cobertura"] == "parcial" for p in validos): return "L"
    return "R"

derivada = {a: marca_derivada(ps) for a, ps in pares_h2.items()}
c_h1 = Counter(h1.cell(r, 4).value for r in filas1)
c_h2 = Counter(derivada.values())
print("Hoja 1:", dict(sorted(c_h1.items())), "| derivado de la Hoja 2:", dict(sorted(c_h2.items())))
print("pares en la Hoja 2:", len(filas2), "(entrada:", len(entrada.pares), ") | artículos:", len(pares_h2))
# 'Sí' solo en A y L, y es el par de mayor confianza entre los de la cobertura que decide la marca
elegidos_ok = True
for a, ps in pares_h2.items():
    si = [p for p in ps if p["Elegido para el papel"] == "Sí"]
    if derivada[a] in ("R", "X", "P"):
        elegidos_ok &= not si
    else:
        cob = "total" if derivada[a] == "A" else "parcial"
        mejor = max(p["Confianza"] for p in ps if p["Cobertura"] == cob)
        elegidos_ok &= len(si) == 1 and si[0]["Cobertura"] == cob and si[0]["Confianza"] == mejor
n_manual = sum(1 for r in filas1 if h1.cell(r, 3).value)
print("elegido 'Sí' coherente (1 en A/L con la mayor confianza; ninguno en R/X/P):", elegidos_ok)
print("filas con manual en Hoja 1:", n_manual, "| A+L:", c_h1["A"] + c_h1["L"], "| columna 3 vacía en R/X/P:", n_manual == c_h1["A"] + c_h1["L"])
print("coincide con conteos_papel (entrada):", dict(c_h1) == {k: v for k, v in conteos_papel(entrada).items() if v})
assert dict(c_h1) == dict(c_h2) and len(filas2) == len(entrada.pares) and elegidos_ok
assert n_manual == c_h1["A"] + c_h1["L"]

Hoja 1: {'A': 11, 'L': 6, 'P': 3, 'R': 6, 'X': 2} | derivado de la Hoja 2: {'A': 11, 'L': 6, 'P': 3, 'R': 6, 'X': 2}
pares en la Hoja 2: 84 (entrada: 84 ) | artículos: 28
elegido 'Sí' coherente (1 en A/L con la mayor confianza; ninguno en R/X/P): True
filas con manual en Hoja 1: 17 | A+L: 17 | columna 3 vacía en R/X/P: True
coincide con conteos_papel (entrada): True


In [10]:
# --- Hoja 2: columnas de docs/10, todos los pares, bloque de la vía 2 (sin tercera hoja)
columnas_docs10 = ["ID par", "Artículo", "Sección manual", "Origen", "Similitud v1", "Similitud v2", "Similitud máxima",
                   "Rango v1", "Rango v2", "Naturaleza", "Cobertura", "Confianza", "Elegido para el papel", "Comentario del juez", "Banderas"]
print("columnas Hoja 2:", enc2)
print("= columnas de docs/10 (sin extras):", enc2 == columnas_docs10)
print("ejemplo Artículo / Sección:", h2.cell(2, 2).value, "//", h2.cell(2, 3).value)
print("\n", h2.cell(fila_via2, 1).value, "| filas:", h2.max_row - fila_via2 - 1, "| hojas:", wb.sheetnames)
for r in range(fila_via2 + 1, min(fila_via2 + 4, h2.max_row + 1)):
    print("  ", [str(h2.cell(r, c).value)[:38] for c in range(1, 6)])
print("elegido / motivo:", Counter(h2.cell(r, col["Elegido para el papel"]).value for r in filas2))
print("banderas:", Counter(h2.cell(r, col["Banderas"]).value for r in filas2))
assert enc2 == columnas_docs10

columnas Hoja 2: ['ID par', 'Artículo', 'Sección manual', 'Origen', 'Similitud v1', 'Similitud v2', 'Similitud máxima', 'Rango v1', 'Rango v2', 'Naturaleza', 'Cobertura', 'Confianza', 'Elegido para el papel', 'Comentario del juez', 'Banderas']
= columnas de docs/10 (sin extras): True
ejemplo Artículo / Sección: N1:Art. 1 | Art. 1 | L1-XVI-cap-III.pdf // M1:4.1 | 4.1 | MOCK-DEMO-01.pdf

 Vía 2 — Controles del manual sin base normativa identificada | filas: 8 | hojas: ['Papel de trabajo', 'Anexo técnico']
   ['ID sección', 'Identificador', 'Documento', 'Ruta', 'Páginas']
   ['M1:III', 'III', 'MOCK-DEMO-01.pdf', 'None', '3']
   ['M1:IV', 'IV', 'MOCK-DEMO-01.pdf', 'None', '3']
elegido / motivo: Counter({'No: cobertura menor': 34, 'No: cobertura nula: ningún par cubre la obligación': 18, 'Sí': 17, 'No: artículo informativo: sin respaldo': 9, 'No: no aplica a la entidad: sin respaldo': 6})
banderas: Counter({None: 80, 'cita_no_verificada, requiere_revision': 3, 'requiere_revision': 1})


## 5 · Seguridad: neutralización de fórmulas (CWE-1236) con casos hostiles
Para cada prefijo (`=`, `+`, `-`, `@`, tabulador, retorno de carro) se antepone a **todo** campo de texto: nombres de documento, rutas, identificadores, títulos, textos de norma y manual, citas, comentarios, elementos faltantes y conclusión.

In [11]:
base = armar_entrada(normas[:1], manual)
resumen_hostil = []
for p in styles.PREFIJOS_FORMULA:
    ruta_h = SALIDA / "papel_hostil.xlsx"
    generar_papel(hostilizar(base, p), ruta_h)
    w = load_workbook(ruta_h)
    formulas = hostiles = sin_prefijo_quote = 0
    for ws in w.worksheets:
        for fila in ws.iter_rows():
            for c in fila:
                if c.data_type == "f": formulas += 1
                if isinstance(c.value, str) and c.value.startswith(p):
                    hostiles += 1
                    if not c.quotePrefix: sin_prefijo_quote += 1
    xml = b"".join(zipfile.ZipFile(ruta_h).read(n) for n in zipfile.ZipFile(ruta_h).namelist() if n.startswith("xl/worksheets/"))
    resumen_hostil.append((repr(p), hostiles, formulas, sin_prefijo_quote, b"<f>" in xml or b"<f " in xml))
    print(f"prefijo {p!r:6}: celdas hostiles {hostiles:4} | celdas con data_type 'f': {formulas} | sin quotePrefix: {sin_prefijo_quote} | <f> en el XML: {resumen_hostil[-1][4]}")
assert all(f == 0 and q == 0 and not x for _, _, f, q, x in resumen_hostil) and all(h > 50 for _, h, *_ in resumen_hostil)
# fórmula clásica de exfiltración, en un campo cualquiera
hh = hostilizar(base, '=HYPERLINK("http://x","clic")+')
generar_papel(hh, ruta_h); w = load_workbook(ruta_h)
print("ejemplo:", repr(w["Papel de trabajo"]["B2"].value), w["Papel de trabajo"]["B2"].data_type)

prefijo '='   : celdas hostiles   97 | celdas con data_type 'f': 0 | sin quotePrefix: 0 | <f> en el XML: False


prefijo '+'   : celdas hostiles   97 | celdas con data_type 'f': 0 | sin quotePrefix: 0 | <f> en el XML: False


prefijo '-'   : celdas hostiles   97 | celdas con data_type 'f': 0 | sin quotePrefix: 0 | <f> en el XML: False
prefijo '@'   : celdas hostiles   97 | celdas con data_type 'f': 0 | sin quotePrefix: 0 | <f> en el XML: False


prefijo '\t'  : celdas hostiles   96 | celdas con data_type 'f': 0 | sin quotePrefix: 0 | <f> en el XML: False
prefijo '\r'  : celdas hostiles   96 | celdas con data_type 'f': 0 | sin quotePrefix: 0 | <f> en el XML: False


ejemplo: 'Evaluación de cumplimiento normativo — =HYPERLINK("http://x","clic")+MOCK-DEMO-01.pdf' s


### Valores no finitos (NaN, inf): celda vacía, sin romper el archivo

In [12]:
from openpyxl import Workbook
wb_nf = Workbook(); ws_nf = wb_nf.active
for i, v in enumerate([float("nan"), float("inf"), float("-inf"), 0.5, 3], 1):
    styles.escribir(ws_nf, i, 1, v)
wb_nf.save(SALIDA / "no_finitos.xlsx")
vals = [load_workbook(SALIDA / "no_finitos.xlsx").active.cell(i, 1).value for i in range(1, 6)]
print("valores leídos:", vals)
assert vals == [None, None, None, 0.5, 3]
# 'Similitud máxima' ignora scores no finitos y 'Origen' dice 'ambas' (docs/10)
par0 = base.pares[0].model_copy(update={"score_v1": float("nan"), "score_v2": 0.8, "origen": {"v1", "v2"}})
e_nan = base.model_copy(update={"pares": [par0] + base.pares[1:]})
t0_ = annex.tabla_pares(e_nan).iloc[0]
print("similitud máxima con NaN y 0.8:", t0_["Similitud máxima"], "| origen:", t0_["Origen"])
assert t0_["Similitud máxima"] == 0.8 and t0_["Origen"] == "ambas"

valores leídos: [None, None, None, 0.5, 3]
similitud máxima con NaN y 0.8: 0.8 | origen: ambas


## 6 · Límite de 32.767 caracteres

In [13]:
art = base.filas[0].articulo
grande = art.model_copy(update={"texto_literal": (art.texto_literal + " ") * (40000 // len(art.texto_literal) + 1)})
f0 = base.filas[0].model_copy(update={"articulo": grande, "comentario": "x" * 40000})
e_big = base.model_copy(update={"filas": [f0] + base.filas[1:], "secciones": {**base.secciones, grande.id: grande}})
generar_papel(e_big, SALIDA / "papel_trunc.xlsx")
w = load_workbook(SALIDA / "papel_trunc.xlsx"); ws = w["Papel de trabajo"]
for col in (2, 5):
    v = ws.cell(13, col).value
    print(f"col {col}: {len(v)} caracteres | termina en: {v[-40:]!r}")
assert len(ws.cell(13, 2).value) <= 32767 and ws.cell(13, 2).value.endswith(f"[texto truncado, ver página {grande.pagina_inicio}]")

col 2: 32767 caracteres | termina en: 'e la Supe\n[texto truncado, ver página 1]'
col 5: 32767 caracteres | termina en: 'xxxxxxxxx\n[texto truncado, ver página 1]'


## 7 · Prueba extra: LibreOffice abre y convierte a PDF

In [14]:
import shutil
if shutil.which("soffice"):   # LibreOffice es opcional: solo comprueba que el Excel se convierte a PDF
    r = subprocess.run(["soffice", "--headless", "--convert-to", "pdf", "--outdir", str(SALIDA), str(ARCHIVO)],
                       capture_output=True, text=True, timeout=300)
    pdf = SALIDA / "papel_l5.pdf"
    print("soffice código:", r.returncode, "|", r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-200:])
    print("PDF:", pdf.exists() and f"{len(pymupdf.open(pdf))} páginas, {pdf.stat().st_size // 1024} KB")
    assert r.returncode == 0 and pdf.exists()
else:
    print("LibreOffice (soffice) no está instalado: se omite la conversión a PDF. El resto del notebook no depende de ella.")

soffice código: 0 | Overwriting: /home/homebridge/Documents/genai/nuevo_comparador/wt-feat-l5-papel/comparador-dbx/output/papel_l5.pdf
PDF: 17 páginas, 466 KB


## 8 · Rendimiento: ~100 artículos × 1 respaldo, `taskset -c 3`, `OMP_NUM_THREADS=1`
Textos reales del proyecto de ley LA/FT contra el manual MOCK (script `scripts/medir_papel.py`; la generación es solo openpyxl + pandas).

In [15]:
env = {**os.environ, "OMP_NUM_THREADS": "1"}
import shutil
afinidad = ["taskset", "-c", "3"] if shutil.which("taskset") else []   # taskset solo existe en Linux
r = subprocess.run([*afinidad, sys.executable, str(RAIZ / "scripts" / "medir_papel.py"), str(NORMAS), "100"],
                   capture_output=True, text=True, env=env, timeout=600, cwd=RAIZ)
medicion = json.loads(r.stdout.strip().splitlines()[-1])
print(json.dumps(medicion, indent=1))
print("\nafinidad de CPU:", afinidad or "no disponible en este SO")

{
 "articulos": 99,
 "pares": 297,
 "respaldos_por_articulo": 1,
 "segundos_generar": 0.37,
 "ram_pico_antes_mb": 132.8,
 "ram_pico_final_mb": 135.6,
 "xlsx_kb": 65,
 "conteos": {
  "A": 39,
  "L": 20,
  "R": 20,
  "X": 10,
  "P": 10
 }
}

el mismo script con taskset: ['taskset', '-c', '3']


## 9 · Lista de chequeo (docs/10 y docs/13)

In [16]:
chequeo = {
    "Abre sin advertencias de reparación (openpyxl, 0 avisos; LibreOffice convierte a PDF)": len(avisos) == 0 and r.returncode is not None and pdf.exists(),
    "Columnas de la matriz en el orden exacto": cabecera == esperadas,
    "Leyenda A/L/R/X/P presente": all(f"{m} = " in ley for m in "ALRXP"),
    "Hoja 2 con todos los pares y bloque de la vía 2 (sin tercera hoja)": len(filas2) == len(entrada.pares) and len(wb.sheetnames) == 2 and fila_via2 > 0,
    "Todo texto de norma/manual es subcadena del fuente o está marcado": not malos,
    "Conteos Hoja 1 = derivables de Hoja 2 (marca re-derivada con docs/09 §5, solo en el notebook)": dict(c_h1) == dict(c_h2) and elegidos_ok,
    "Columna 3 vacía en R, X y P; Hoja 2 solo con las columnas de docs/10": n_manual == c_h1["A"] + c_h1["L"] and enc2 == columnas_docs10,
    "NaN e inf no rompen el archivo (celda vacía)": vals == [None, None, None, 0.5, 3],
    "Cero llamadas al modelo durante generar_papel (cliente y red)": cliente.llamadas_llm == llm0 and not conexiones,
    "Conclusión: 1 llamada (2 con reintento), 120–220 palabras, sin cifras ajenas, con prefijo": cliente.llamadas_llm in (1, 2) and h1["B7"].value.startswith(PREFIJO_BORRADOR),
    "Ninguna celda es fórmula con prefijos hostiles (=,+,-,@,TAB,CR)": all(f == 0 and q == 0 for _, _, f, q, _ in resumen_hostil),
    "Truncado a 32.767 con aviso": len(ws.cell(13, 2).value) <= 32767,
    "Rendimiento medido (100 artículos)": medicion["articulos"] >= 99,
}
for k, v in chequeo.items(): print("✅" if v else "❌", k)
assert all(chequeo.values())
print("\nL5: CRITERIO CUMPLIDO" if all(chequeo.values()) else "\nL5: NO CUMPLIDO")

✅ Abre sin advertencias de reparación (openpyxl, 0 avisos; LibreOffice convierte a PDF)
✅ Columnas de la matriz en el orden exacto
✅ Leyenda A/L/R/X/P presente
✅ Hoja 2 con todos los pares y bloque de la vía 2 (sin tercera hoja)
✅ Todo texto de norma/manual es subcadena del fuente o está marcado
✅ Conteos Hoja 1 = derivables de Hoja 2 (marca re-derivada con docs/09 §5, solo en el notebook)
✅ Columna 3 vacía en R, X y P; Hoja 2 solo con las columnas de docs/10
✅ NaN e inf no rompen el archivo (celda vacía)
✅ Cero llamadas al modelo durante generar_papel (cliente y red)
✅ Conclusión: 1 llamada, 120–220 palabras, sin cifras ajenas, con prefijo
✅ Ninguna celda es fórmula con prefijos hostiles (=,+,-,@,TAB,CR)
✅ Truncado a 32.767 con aviso
✅ Rendimiento medido (100 artículos)

L5: CRITERIO CUMPLIDO
